# RepCount Part-A — Data Preparation

This notebook is the **data preparation pipeline** separated from EDA.

**Why this notebook exists**
- EDA should diagnose issues; preparation should apply deterministic fixes and export model-ready data.
- Keeping them separate avoids accidental metric drift and improves reproducibility.

**Current decisions in this version**
- Normalize known typo labels.
- Apply manual review decisions from `others` inspection.
- Keep `pommelhorse`.
- Keep `battle_rope` temporarily as domain/class-size policy.
- Keep `rowing_erg` 


**Next steps after running this notebook**
- Use exported cleaned CSVs in model training configs.
- Track this policy in experiment metadata.
- Re-run when label rules or class scope changes.


## Project Goal and Success Metrics

### Goal
**Real-Time Rep Counter via Human Pose Estimation**

### Core pipeline objective
1. **Detect the person** in each frame (or track the main subject).
2. **Detect movement dynamics** from pose over time.
3. **Count completed repetitions** robustly in real time.

### Primary metric
- **Rep Count MAE (Mean Absolute Error)** per video/session.

### Required supporting metrics
- **Per-class MAE** (to reveal class-specific weakness and imbalance effects).
- **Real-time performance** (latency/FPS for deployment readiness).

### Optional reporting metrics
- **RMSE** on rep count.
- **Within-1 rep accuracy**.



In [1]:
import os
import pandas as pd
import numpy as np

print('Libraries loaded')


Libraries loaded


## 1) Paths and Config

**Why this section**
- Centralizes file locations and policy switches so the pipeline is easy to audit and change.

**Decision(s)**
- Input comes from raw LLSP annotation CSVs.
- Output is written to `annotation_cleaned/`.
- `EXCLUDE_TYPES` controls temporary domain filtering.

**Next step(s)**
- Update paths if data storage changes.
- Update `EXCLUDE_TYPES` only with documented rationale.


In [2]:
TRAIN_PATH = '../../Data/LLSP/annotation/train.csv'
VALID_PATH = '../../Data/LLSP/annotation/valid.csv'

OUTPUT_DIR = '../../Data/LLSP/annotation_cleaned'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Toggle if you later want to exclude whole classes from training domain
EXCLUDE_TYPES = ['battle_rope', 'pommelhorse']  # temporary domain filter

# Policy note: we remove reviewed rowing_erg candidates because class size < 30



## 2) Load Splits

**Why this section**
- Establishes a single, explicit source of truth for train/valid/test before transformation.

**Decision(s)**
- Preserve split identity in each dataframe.
- Keep all raw columns intact (including temporal `L*` fields).

**Next step(s)**
- Verify row counts against expected dataset totals before any cleaning.


In [3]:
def load_split(path, split_name):
    df = pd.read_csv(path)
    if df.columns[0] in ['Unnamed: 0', '']:
        df = df.rename(columns={df.columns[0]: 'index'})
    df['split'] = split_name
    return df

train_df = load_split(TRAIN_PATH, 'train')
valid_df = load_split(VALID_PATH, 'valid')

df_all_raw = pd.concat([train_df, valid_df], ignore_index=True)

print('Loaded rows:')
print(f"  train={len(train_df)}, valid={len(valid_df)}, total={len(df_all_raw)}")
print(f"Raw unique train labels ({train_df['type'].nunique()}):", sorted(train_df['type'].dropna().unique()))



Loaded rows:
  train=758, valid=131, total=889
Raw unique train labels (16): ['battle_rope', 'bench_pressing', 'benchpressing', 'front_raise', 'frontraise', 'jump_jack', 'jumpjacks', 'others', 'pommelhorse', 'pull_up', 'pullups', 'push_up', 'pushups', 'situp', 'squant', 'squat']


/var/folders/7q/522tn7j14w119bz944bz8qy80000gn/T/ipykernel_38845/3425875354.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['split'] = split_name
/var/folders/7q/522tn7j14w119bz944bz8qy80000gn/T/ipykernel_38845/3425875354.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['split'] = split_name


## 3) Typo Normalization

**Why this section**
- Label variants fragment classes and corrupt class distributions, sampling, and metrics.

**Decision(s)**
- Apply a fixed typo map (e.g., `squant -> squat`, `situp -> sit_up`, etc.).
- Use the same normalization across train/valid/test for consistency.

**Next step(s)**
- If new variants are found, add them here and re-run full prep + EDA checks.


In [4]:
TYPO_MAP = {
    'squant'        : 'squat',
    'frontraise'    : 'front_raise',
    'benchpressing' : 'bench_pressing',
    'jumpjacks'     : 'jump_jacks',
    'jump_jack'     : 'jump_jacks',
    'situp'         : 'sit_up',
    'pullsup'       : 'pull_up',
    'pullsups'      : 'pull_up',
    'pullups'       : 'pull_up',
    'pushups'       : 'push_up',
    'push_ups'      : 'push_up',
}

for d in [train_df, valid_df]:
    d['type'] = d['type'].replace(TYPO_MAP)

print('After typo-clean, train labels:', sorted(train_df['type'].dropna().unique()))

# Snapshot class counts right after typo-normalization (before manual/exclusion decisions)
train_counts_after_typo = train_df['type'].value_counts().sort_values(ascending=False).copy()
valid_counts_after_typo = valid_df['type'].value_counts().sort_values(ascending=False).copy()

print('\nBaseline train distribution (after typo-fix, before decisions):')
print(train_counts_after_typo.to_string())



After typo-clean, train labels: ['battle_rope', 'bench_pressing', 'front_raise', 'jump_jacks', 'others', 'pommelhorse', 'pull_up', 'push_up', 'sit_up', 'squat']

Baseline train distribution (after typo-fix, before decisions):
type
squat             101
pull_up            94
front_raise        93
bench_pressing     93
sit_up             93
push_up            89
jump_jacks         76
pommelhorse        69
others             37
battle_rope        13


## 4) Manual Review Decisions (`others` + corrections)

**Why this section**
- Human review resolves ambiguous/mislabeled videos that automated rules cannot fix.

**Decision(s)**
- `stu6_11.mp4 -> squat` (manual inspection correction).
- Remove reviewed `others` videos confirmed as ambiguous/out-of-domain/noisy.
- Remove reviewed `rowing_erg` candidates because class size is below threshold (<30), which is too small for stable class-level training.
- Keep decision logic explicit via `RELABEL_MAP` and `REMOVE_LIST`.

**Next step(s)**
- Add each new manual decision with evidence (PDF/frame review note).
- If more `rowing_erg` data is collected later, reverse this removal policy and re-run prep + weights.



In [ ]:
RELABEL_MAP = {
    # confirmed manual correction
    'stu6_11.mp4': 'squat',
    'stu11_10.mp4': 'squat', 'stu11_11.mp4': 'squat', 'stu11_12.mp4': 'squat', 'stu11_13.mp4': 'squat', 'stu11_7.mp4': 'squat', 'stu11_8.mp4': 'squat',
    'stu11_9.mp4': 'squat', 'stu12_0.mp4': 'squat', 'stu12_7.mp4': 'squat', 'stu13_0.mp4': 'squat', 'stu13_4.mp4': 'squat',
}

REMOVE_LIST = [
    # reviewed others removed for quality/domain reasons
    'stu1_28.mp4', 'stu1_29.mp4', 'stu11_0.mp4', 'stu11_1.mp4', 'stu12_2.mp4', 'stu12_6.mp4', 'stu12_8.mp4',
    'stu12_1.mp4', 'stu12_5.mp4', 'stu12_9.mp4', 'stu12_10.mp4', 'stu13_1.mp4', 'stu13_5.mp4', 'stu13_6.mp4',
    'stu11_4.mp4', 'stu11_5.mp4', 'stu11_6.mp4', 'stu12_3.mp4', 'stu12_4.mp4', 'stu13_2.mp4', 'stu13_3.mp4',
    'stu5_28.mp4', 'stu10_32.mp4', 'stu10_33.mp4', 'stu11_2.mp4', 'stu11_3.mp4',


]

def normalize_video_name(x):
    x = str(x).strip()
    return x if x.endswith('.mp4') else f'{x}.mp4'

relabel_norm = {normalize_video_name(k): v for k, v in RELABEL_MAP.items()}
remove_set = {normalize_video_name(x) for x in REMOVE_LIST}

# Apply manual curation ONLY to train/valid
affected_rows = {'train': {'relabel': 0, 'removed': 0}, 'valid': {'relabel': 0, 'removed': 0}}
for d, name in [(train_df, 'train'), (valid_df, 'valid')]:
    # relabel
    relabel_hits = 0
    for video_name, new_label in relabel_norm.items():
        mask = d['name'] == video_name
        if mask.any():
            relabel_hits += int(mask.sum())
            d.loc[mask, 'type'] = new_label

    # remove
    before = len(d)
    d.drop(d[d['name'].isin(remove_set)].index, inplace=True)
    d.reset_index(drop=True, inplace=True)
    removed = before - len(d)

    affected_rows[name]['relabel'] = relabel_hits
    affected_rows[name]['removed'] = removed
    print(f"{name:<5} relabeled rows: {relabel_hits}")
    print(f"{name:<5} removed rows: {removed}")



train relabeled rows: 1
train removed rows: 37
valid relabeled rows: 0
valid removed rows: 0


## 5) Optional Domain Class Exclusion

**Why this section**
- Some classes may be valid in dataset scope but out-of-scope for deployment domain.

**Decision(s)**
- Apply `EXCLUDE_TYPES` as a controlled domain filter.
- Current default excludes `battle_rope` due to very low sample count and temporary domain focus.

**Next step(s)**
- Re-enable `battle_rope` once additional labeled data is collected.
- Keep this flag synchronized with training/evaluation configs and report notes.



In [6]:
if EXCLUDE_TYPES:
    excluded_rows = {'train': 0, 'valid': 0}
    for d, name in [(train_df, 'train'), (valid_df, 'valid')]:
        before = len(d)
        d.drop(d[d['type'].isin(EXCLUDE_TYPES)].index, inplace=True)
        d.reset_index(drop=True, inplace=True)
        excluded_rows[name] = before - len(d)
        print(f"{name:<5} excluded class rows: {excluded_rows[name]}")
else:
    excluded_rows = {'train': 0, 'valid': 0}
    print('No class exclusions applied')



train excluded class rows: 82
valid excluded class rows: 18


## Dataset Finalization Decisions

### Why this section exists
This section records **formal data-finalization policy** before modeling so that training artifacts are reproducible and reviewable.

### Final decisions applied in this notebook
1. **Scope policy**
   - This notebook processes **train + valid only**.
   - Test split is intentionally excluded from curation decisions to avoid evaluation leakage.

2. **Label normalization**
   - Known typo variants are normalized through `TYPO_MAP`.
   - Purpose: ensure semantic label consistency and prevent duplicated classes due to naming noise.

3. **Manual relabeling**
   - `RELABEL_MAP` contains confirmed manual corrections from visual inspection.
   - Current confirmed correction: `stu6_11.mp4 -> squat`.

4. **Manual removals**
   - `REMOVE_LIST` includes videos removed after inspection where labels/domain fit were not acceptable.
   - Includes reviewed `others` and `rowing_erg` candidates flagged during curation.

5. **Class exclusions**
   - `EXCLUDE_TYPES = ['battle_rope', 'pommelhorse']`.
   - Rationale:
     - `battle_rope`: too few samples for reliable training signal.
     - `pommelhorse`: low domain relevance for current target exercise set.

### Imbalance handling implication
After these decisions, imbalance is recalculated and class/sample weights are generated from the **final train split**.

### Reproducibility outputs
This notebook exports:
- cleaned train/valid CSVs,
- class/sample weighting files,
- `decisions_manifest.json` capturing policy + applied decisions.

### Next steps
- Train baseline model with weighted loss and balanced sampling.
- Track per-class metrics (not only global metrics).
- Revisit removed/excluded classes only if domain scope changes or new data is collected.

### Class-level review log
| class        | status  | note                                      |
|--------------|---------|-------------------------------------------|
| others       | Removed | OOD: soccer, rowing sport, hammer         |
| rowing_erg   | Pending | 11 samples — search for more first        |



## Data Leakage Checks

### Why this section exists
This section verifies that train/validation boundaries are clean and that preprocessing decisions are not introducing leakage.

### Leakage checks performed
1. **Exact split overlap by `name`**
   - Detects videos appearing in both train and validation.
2. **Exact row overlap (`type`, `name`, `count`)**
   - Detects duplicated labeled records across splits.
3. **Near-duplicate base-name overlap**
   - Detects likely related clips such as `train123.mp4` and `train123_1.mp4` crossing splits.
4. **Curation policy scope checks**
   - Confirms relabel/remove lists do not unexpectedly hit validation beyond intended policy.
5. **Summary verdict**
   - Prints PASS/WARN flags for quick audit.

### Interpretation
- Any non-zero overlap should be reviewed before modeling.
- Near-duplicate overlap is a warning signal and may require manual review.
- This section is a guardrail, not proof of zero leakage for all hidden metadata.



In [7]:
import re

# --- helper functions ---
def _safe_series(df, col):
    return df[col].astype(str).str.strip() if col in df.columns else pd.Series(dtype=str)

def _normalize_name(s):
    s = str(s).strip()
    return s.lower()

def _base_video_name(name):
    # Remove extension, then common clip suffix patterns (e.g., _1, -2)
    n = str(name).strip().lower()
    n = re.sub(r'\.mp4$', '', n)
    n = re.sub(r'([_-]clip)?[_-]?\d+$', '', n)
    return n

# --- 1) exact overlap by video name ---
train_names = set(_safe_series(train_df, 'name').map(_normalize_name))
valid_names = set(_safe_series(valid_df, 'name').map(_normalize_name))
name_overlap = sorted(train_names & valid_names)

print('Leakage check 1/5 — exact name overlap (train vs valid):')
print(f'  overlap_count = {len(name_overlap)}')
if len(name_overlap) > 0:
    print('  sample_overlaps:', name_overlap[:20])

# --- 2) exact row overlap by key columns ---
key_cols = [c for c in ['type', 'name', 'count'] if c in train_df.columns and c in valid_df.columns]
if key_cols:
    train_keys = set(map(tuple, train_df[key_cols].astype(str).values.tolist()))
    valid_keys = set(map(tuple, valid_df[key_cols].astype(str).values.tolist()))
    row_overlap = train_keys & valid_keys
else:
    row_overlap = set()

print('Leakage check 2/5 — exact row overlap (type,name,count):')
print(f'  overlap_count = {len(row_overlap)}')

# --- 3) near-duplicate overlap using base names ---
train_base = set(_safe_series(train_df, 'name').map(_base_video_name))
valid_base = set(_safe_series(valid_df, 'name').map(_base_video_name))
base_overlap = sorted(train_base & valid_base)

print('Leakage check 3/5 — near-duplicate base-name overlap:')
print(f'  overlap_count = {len(base_overlap)}')
if len(base_overlap) > 0:
    print('  sample_base_overlaps:', base_overlap[:20])

# --- 4) policy-scope checks for curation lists ---
# This notebook policy is train+valid scope; test is out-of-scope here.
relabel_norm_local = {str(k).strip().lower() if str(k).strip().lower().endswith('.mp4') else f"{str(k).strip().lower()}.mp4": v
                     for k, v in RELABEL_MAP.items()} if 'RELABEL_MAP' in globals() else {}
remove_set_local = {str(x).strip().lower() if str(x).strip().lower().endswith('.mp4') else f"{str(x).strip().lower()}.mp4"
                    for x in REMOVE_LIST} if 'REMOVE_LIST' in globals() else set()

valid_name_norm = set(_safe_series(valid_df, 'name').map(_normalize_name))
valid_relabel_hits = sorted([n for n in relabel_norm_local if n in valid_name_norm])
valid_remove_hits = sorted([n for n in remove_set_local if n in valid_name_norm])

print('Leakage check 4/5 — curation policy scope audit (validation hits):')
print(f'  relabel_hits_in_valid = {len(valid_relabel_hits)}')
print(f'  remove_hits_in_valid  = {len(valid_remove_hits)}')

# --- 5) verdict ---
exact_overlap_pass = len(name_overlap) == 0 and len(row_overlap) == 0
near_dup_warn = len(base_overlap) > 0

print('Leakage check 5/5 — summary verdict:')
print('  exact_split_overlap_pass =', exact_overlap_pass)
print('  near_duplicate_warning   =', near_dup_warn)

if exact_overlap_pass and not near_dup_warn:
    print('LEAKAGE STATUS: PASS (no exact/near-duplicate signals from current checks).')
elif exact_overlap_pass and near_dup_warn:
    print('LEAKAGE STATUS: WARN (no exact overlap, but near-duplicate candidates exist).')
else:
    print('LEAKAGE STATUS: FAIL (exact overlap detected; fix before modeling).')



Leakage check 1/5 — exact name overlap (train vs valid):
  overlap_count = 0
Leakage check 2/5 — exact row overlap (type,name,count):
  overlap_count = 0
Leakage check 3/5 — near-duplicate base-name overlap:
  overlap_count = 13
  sample_base_overlaps: ['stu1', 'stu10', 'stu2', 'stu3', 'stu4', 'stu5', 'stu6', 'stu7', 'stu8', 'stu9', 'test', 'train', 'val']
Leakage check 4/5 — curation policy scope audit (validation hits):
  relabel_hits_in_valid = 0
  remove_hits_in_valid  = 0
Leakage check 5/5 — summary verdict:
  exact_split_overlap_pass = True
  near_duplicate_warning   = True
LEAKAGE STATUS: WARN (no exact overlap, but near-duplicate candidates exist).


## 5.1) Decision Impact Check (Before vs After)

**Why this section**
- Any class refactor (relabel/remove/exclude) changes the training distribution.
- Imbalance must be recomputed on the **final train set**, not on pre-clean counts.

**Decision(s)**
- Compare train class counts before decisions (`train_counts_after_typo`) vs final (`train_df`).
- Report train imbalance ratio before and after.

**Next step(s)**
- Use post-decision imbalance numbers for sampler/loss configuration.
- Re-run this section whenever decision lists or exclusions change.



In [8]:
final_train_counts = train_df['type'].value_counts().sort_values(ascending=False)

cmp = pd.concat([
    train_counts_after_typo.rename('before_decisions'),
    final_train_counts.rename('after_decisions')
], axis=1).fillna(0).astype(int)
cmp['delta'] = cmp['after_decisions'] - cmp['before_decisions']

print('Class count impact (train):')
print(cmp.to_string())

before_ratio = train_counts_after_typo.max() / train_counts_after_typo.min()
after_ratio = final_train_counts.max() / final_train_counts.min()
print(f"\nImbalance ratio (train) before decisions: {before_ratio:.2f}x")
print(f"Imbalance ratio (train) after decisions : {after_ratio:.2f}x")



Class count impact (train):
                before_decisions  after_decisions  delta
type                                                    
squat                        101              102      1
pull_up                       94               94      0
front_raise                   93               92     -1
bench_pressing                93               93      0
sit_up                        93               93      0
push_up                       89               89      0
jump_jacks                    76               76      0
pommelhorse                   69                0    -69
others                        37                0    -37
battle_rope                   13                0    -13

Imbalance ratio (train) before decisions: 7.77x
Imbalance ratio (train) after decisions : 1.34x


## 6) Build Clean Analysis Frame and Audit

**Why this section**
- Confirms final data integrity before training: split sizes, labels, and unresolved classes.

**Decision(s)**
- Build `df_tv = train + valid` for analysis/training context.
- Drop rows with missing `count` in train+valid.
- Verify `others`/`rowing_erg` status post-decisions.

**Next step(s)**
- Stop training if audit checks fail (unexpected labels, unresolved classes, null targets).


In [9]:
df_tv = pd.concat([train_df, valid_df], ignore_index=True)
missing_count = int(df_tv['count'].isna().sum())
if missing_count > 0:
    print(f"Dropping {missing_count} rows with missing count from train+valid")

df_tv = df_tv.dropna(subset=['count']).copy()

print('\nFinal split sizes:')
print(f"  train={len(train_df)}, valid={len(valid_df)}")
print(f"  train+valid (count non-null)={len(df_tv)}")

print('\nFinal train label distribution:')
print(train_df['type'].value_counts().to_string())

print('\nRemaining `others` in train+valid:', int((df_tv['type']=='others').sum()))
print('Remaining `rowing_erg` in train+valid:', int((df_tv['type']=='rowing_erg').sum()))



Dropping 1 rows with missing count from train+valid

Final split sizes:
  train=639, valid=113
  train+valid (count non-null)=751

Final train label distribution:
type
squat             102
pull_up            94
bench_pressing     93
sit_up             93
front_raise        92
push_up            89
jump_jacks         76

Remaining `others` in train+valid: 0
Remaining `rowing_erg` in train+valid: 0


## 7) Class Imbalance Weights (for training)

**Why this section**
- Imbalance biases optimization toward majority classes.

**Decision(s)**
- Compute inverse-frequency class weights from **final cleaned train set**.
- Normalize weights to mean ~1 for stable scaling.
- Do not compute weights from raw/pre-decision labels.

**Next step(s)**
- Use these weights in weighted loss and/or balanced sampler.
- Recompute weights whenever labels, removals, or exclusions change.



In [10]:
class_counts = train_df['type'].value_counts().sort_index()

# Inverse-frequency weights (normalized to mean=1)
inv = 1.0 / class_counts
class_weights = inv / inv.mean()

weights_df = pd.DataFrame({
    'count': class_counts,
    'inv_freq_weight': class_weights.round(6)
}).sort_values('count', ascending=False)

print(weights_df.to_string())


                count  inv_freq_weight
type                                  
squat             102         0.888830
pull_up            94         0.964475
bench_pressing     93         0.974846
sit_up             93         0.974846
front_raise        92         0.985442
push_up            89         1.018659
jump_jacks         76         1.192903


## 8) Class Imbalance Strategy (Recommended Defaults)

**Why this section**
- Cleaning decisions reduce major imbalance, but minority classes still need robust training setup.

**Decision(s)**
- Use **class-weighted loss** from final cleaned train counts.
- Use a **weighted sampler** so minority classes appear more often in mini-batches.
- Monitor **per-class MAE** and macro-averaged metrics in validation.

**Default strategy for this project**
1. Loss weighting: inverse-frequency weights (already computed in this notebook).
2. Sampling: `WeightedRandomSampler` (PyTorch) using per-sample class weights.
3. Evaluation: report `overall MAE`, `macro MAE`, and `per-class MAE` every run.
4. Escalation path: if minority performance remains poor, collect more real data before synthetic generation.

**Next step(s)**
- Export sampler weights and class weights with cleaned annotations.
- Wire these outputs into your training dataloader + loss config.



In [11]:
# Build per-sample train weights for WeightedRandomSampler-style training
# (each sample gets the weight of its class)

train_for_sampler = train_df[['name', 'type']].copy()
weight_map = class_weights.to_dict()  # from Section 7
train_for_sampler['sample_weight'] = train_for_sampler['type'].map(weight_map).astype(float)

# quick sanity check
print('Sampler weight sanity check (first 10 rows):')
print(train_for_sampler.head(10).to_string(index=False))

print('\nSampler weight summary by class:')
print(train_for_sampler.groupby('type')['sample_weight'].mean().sort_values(ascending=False).to_string())



Sampler weight sanity check (first 10 rows):
        name           type  sample_weight
train951.mp4    front_raise       0.985442
train952.mp4    front_raise       0.985442
test1463.mp4        pull_up       0.964475
test2340.mp4          squat       0.888830
 stu5_11.mp4    front_raise       0.985442
  stu9_2.mp4 bench_pressing       0.974846
 stu6_18.mp4     jump_jacks       1.192903
 stu4_14.mp4    front_raise       0.985442
  stu8_3.mp4 bench_pressing       0.974846
 stu4_54.mp4         sit_up       0.974846

Sampler weight summary by class:
type
jump_jacks        1.192903
push_up           1.018659
front_raise       0.985442
bench_pressing    0.974846
sit_up            0.974846
pull_up           0.964475
squat             0.888830


## 9) Export Cleaned CSVs

**Why this section**
- Training should consume frozen, versioned cleaned artifacts rather than raw files.

**Decision(s)**
- Export cleaned train/valid/test plus train class weights and sampler weights.

**Next step(s)**
- Point training pipeline paths to `annotation_cleaned/` outputs.
- Tag outputs with experiment/version metadata if you move to MLOps tracking.


In [12]:
import json
from datetime import datetime, timezone

train_out = os.path.join(OUTPUT_DIR, 'train_cleaned.csv')
valid_out = os.path.join(OUTPUT_DIR, 'valid_cleaned.csv')
weights_out = os.path.join(OUTPUT_DIR, 'class_weights_train.csv')
sampler_out = os.path.join(OUTPUT_DIR, 'train_sample_weights.csv')
manifest_out = os.path.join(OUTPUT_DIR, 'decisions_manifest.json')

train_df.to_csv(train_out, index=False)
valid_df.to_csv(valid_out, index=False)
weights_df.reset_index().rename(columns={'index':'type'}).to_csv(weights_out, index=False)
train_for_sampler.to_csv(sampler_out, index=False)

manifest = {
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'input_paths': {
        'train': TRAIN_PATH,
        'valid': VALID_PATH,
    },
    'output_paths': {
        'train_cleaned': train_out,
        'valid_cleaned': valid_out,
        'class_weights_train': weights_out,
        'train_sample_weights': sampler_out,
    },
    'policy': {
        'scope': 'train_valid_only',
        'typo_map_applied_to': ['train', 'valid'],
        'manual_relabel_applied_to': ['train', 'valid'],
        'remove_list_applied_to': ['train', 'valid'],
        'exclude_types_applied_to': ['train', 'valid'],
        'test_policy': 'not handled in this notebook',
    },
    'decisions': {
        'exclude_types': EXCLUDE_TYPES,
        'relabel_map': RELABEL_MAP,
        'remove_list': REMOVE_LIST,
    },
    'changes_summary': {
        'manual_relabel_rows': affected_rows,
        'excluded_rows': excluded_rows,
    },
    'row_counts': {
        'train_cleaned': int(len(train_df)),
        'valid_cleaned': int(len(valid_df)),
    },
}

with open(manifest_out, 'w', encoding='utf-8') as f:
    json.dump(manifest, f, ensure_ascii=True, indent=2)

print('Saved:')
print(' ', train_out)
print(' ', valid_out)
print(' ', weights_out)
print(' ', sampler_out)
print(' ', manifest_out)



Saved:
  ../../Data/LLSP/annotation_cleaned/train_cleaned.csv
  ../../Data/LLSP/annotation_cleaned/valid_cleaned.csv
  ../../Data/LLSP/annotation_cleaned/class_weights_train.csv
  ../../Data/LLSP/annotation_cleaned/train_sample_weights.csv
  ../../Data/LLSP/annotation_cleaned/decisions_manifest.json


## 10) Data Preparation Summary

### What was done
- Standardized noisy labels using `TYPO_MAP`.
- Applied manual relabel/removal decisions from inspection (`RELABEL_MAP`, `REMOVE_LIST`).
- Applied class-domain exclusions (`EXCLUDE_TYPES`) for train/valid modeling scope.
- Recomputed post-decision class distribution and imbalance ratio.
- Generated class/sample weighting artifacts for optional imbalance mitigation.
- Exported cleaned train/valid datasets and a reproducibility manifest.

### Final modeling scope
- This notebook is **train+valid only**.
- Test is intentionally out of scope for this stage.

### Exported artifacts
- `train_cleaned.csv`
- `valid_cleaned.csv`
- `class_weights_train.csv`
- `train_sample_weights.csv`
- `decisions_manifest.json`

### Readiness check
- Labels normalized and curated.
- Split-boundary leakage checks added (train vs valid).
- Artifacts versioned via manifest.

### Next step
Proceed to model development in:
- `CV_Image_pose_detection/artifacts/modeling/Model_Training_01.ipynb`

